# 📝 과제 01: 문장 의도 7종을 트랜스포머 인코더로 분류하기

## 만들 결과

짧은 한국어 문장이 평서, 질문, 명령 같은 의도 클래스 7개 가운데 무엇인지 맞히는 분류 모델을 만듭니다. 교안 01의 순서(토큰화, 임베딩, 어텐션, 위치 인코딩, 패딩 마스크, 인코더 블록, 분류 학습)를 영화 리뷰 대신 이 데이터에 적용합니다.

<img src="images/40_gwaje01_overview.png" width="1050" alt="제목은 '과제 01: 문장 의도 7종을 트랜스포머 인코더로 분류하기'입니다. 문항 1~9마다 구역 하나가 3줄 3칸으로 놓여 있습니다. 1 문자 토큰과 어휘 사전은 내, 일, 공백, 오가 토큰 ID 31, 5, 3, 22가 되는 장면과 '어휘 크기 1,104 (학습 문장으로만)', 2 max_len과 패딩은 [CLS], 내, ..., 줘, [PAD], ..., [PAD] 줄과 '[CLS] 1 + 문자 12 + [PAD] 27 = 40', 'max_len은 길이 분포를 보고 정합니다', 3 토큰 임베딩은 '2번 위치 일'과 '7번 위치 일'이 같은 '임베딩 행렬 5번 행'을 꺼내는 장면과 '같은 문자는 임베딩 행렬의 같은 행을 꺼냅니다'입니다. 4 어텐션과 위치 인코딩은 내, 일, 오를 오, 일, 내로 뒤집는 화살표 아래에 '위치 인코딩 없음: 토큰마다 출력이 같음'(흰 칩)과 '위치 인코딩 더함: 출력이 달라짐'(초록 칩)이 있습니다. 5 패딩 마스크는 '[CLS] 행의 어텐션 가중치' 막대 그림으로, [CLS], 내, ..., 줘 위에는 보라 막대가 서 있고 [PAD] 세 칸 위에는 빨간 점선 빈 막대에 빨간 X가 그어져 있으며, 아래에 'key_padding_mask로 [PAD] 가리기', '[PAD] 어텐션 가중치 합: 마스크 없이 약 0.63', '-> 마스크와 함께 0.0'이 적혀 있습니다. 6 인코더 층 쌓기는 인코더 층 1~3과 '× 3', '층 하나 12,704개 -> 3층 38,112개', 7 의도 분류 모델은 [CLS]에서 0~6 로짓 막대 7개(2번이 주황)와 '[CLS] 벡터 -> 의도 클래스 7개의 로짓', 8 학습은 손실과 정확도 곡선(학습 파랑, 검증 주황), 9 평가와 학습 전후 비교는 혼동 행렬과 '학습 전 예측 -> 학습 후 예측', '의도별 정확도와 헷갈린 의도 찾기'입니다.">

- 학습 문장으로 문자 단위 어휘 사전(어휘 크기 1,104)을 만듭니다.
- 문장 길이 분포로 `max_len`(40)을 정하고 문장을 텐서로 바꿉니다.
- `nn.Embedding`, `positional_encoding`, `nn.MultiheadAttention`으로 같은 문자 토큰의 임베딩 벡터, 순서를 뒤집은 입력, 패딩 마스크를 확인합니다.
- `nn.TransformerEncoder`로 인코더 층 3개를 쌓고 파라미터 수를 셉니다(층 하나 12,704개).
- `TransformerClassifier`를 8에폭 학습하고, 의도별 정확도, 혼동 행렬, 학습 전·후 예측을 비교합니다.

## 준비: 데이터와 제공 코드

- **자료:** `data/과제01/intent_train.csv`(20,000문장), `intent_test.csv`(3,000문장). 한국어 문장 의도 말뭉치 [3i4k](https://github.com/warnikchow/3i4k)(Cho et al., 2018, CC BY-SA 4.0)에서 골랐고, 두 파일에 같은 문장은 없습니다. 열은 `문장`, `의도`(0~6), `의도 이름`입니다.
- **의도 클래스 7개:** 0 조각(끝나지 않은 말), 1 평서, 2 질문, 3 명령, 4 수사 의문(질문 꼴이지만 묻지 않는 말), 5 수사 명령(명령 꼴이지만 시키지 않는 말), 6 억양 의존(`내일 비 와`처럼 말의 높낮이로 의도가 갈리는 말)입니다.
  - 말하는 사람이 원하는 것이 정보이면 질문, 행동이면 명령입니다. `내일 오후 일정 알려줘`는 `~줘`로 끝나도 질문입니다.
- **제공 코드:** 아래 준비 셀이 만듭니다. 함수는 교안 01과 같은 코드입니다.

| 이름 | 내용 |
|---|---|
| `intent_train`, `intent_test`, `intent_names` | 두 자료를 읽은 DataFrame과 의도 이름 7개(클래스 인덱스 0~6 순서) |
| `min_count` | 어휘 사전 기준: 학습 문장에서 3번 이상 나온 문자만 어휘 사전에 넣습니다(교안 3-2) |
| `encode`, `make_tensor` | 문자 토큰의 토큰 ID 리스트, [CLS]를 붙여 자르고 패딩한 토큰 ID 텐서(교안 3-2, 4절) |
| `count_parameters` | 파라미터 수(교안 9-3) |
| `positional_encoding` | sin, cos 위치 인코딩 행렬(교안 7-2) |
| `TransformerClassifier`, `evaluate`, `train_classifier`, `loss_fn` | 분류 모델, 평가, 학습 함수, 교차 엔트로피 손실(교안 10절) |

- **sklearn 함수:** 9번 작성 셀이 `from sklearn.metrics import confusion_matrix, recall_score`로 혼동 행렬과 재현율 함수를 불러옵니다.

- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행합니다. 뒤 문항이 앞 문항의 변수를 쓰므로 1번부터 차례로 풉니다.

In [ ]:
import platform
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 단원 폴더의 모델 파일(encoder_models.py)을 import 할 수 있게 경로에 더합니다
sys.path.append(str(material_dir))

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 문장 의도 자료: 문장, 의도(0~6), 의도 이름
intent_train = pd.read_csv(data_dir / "과제01" / "intent_train.csv")
intent_test = pd.read_csv(data_dir / "과제01" / "intent_test.csv")
intent_names = ["조각", "평서", "질문", "명령", "수사 의문", "수사 명령", "억양 의존"]
print("학습용 DataFrame shape:", intent_train.shape, "| 테스트용 DataFrame shape:", intent_test.shape)
display(intent_train.head())
display(intent_train["의도 이름"].value_counts())

In [ ]:
# 교안 01의 모델 클래스와 함수입니다. 실행만 하고 넘어갑니다.
from torch.utils.data import TensorDataset, DataLoader

from encoder_models import TransformerClassifier, positional_encoding

min_count = 3  # 학습 문장에서 3번 이상 나온 문자만 어휘 사전에 넣습니다
loss_fn = nn.CrossEntropyLoss()  # evaluate와 train_classifier가 쓰는 교차 엔트로피 손실


def encode(text, token_to_id):
    """문장을 문자로 자르고, 문자마다 토큰 ID를 찾습니다. 어휘 사전에 없는 문자는 [UNK]의 토큰 ID입니다."""
    ids = []
    for token in list(text):
        if token in token_to_id:
            ids.append(token_to_id[token])
        else:
            ids.append(token_to_id["[UNK]"])
    return ids


def make_tensor(texts, token_to_id, max_len):
    """문장마다 [CLS] + 문자 토큰 ID 시퀀스를 만들고, max_len에 맞춰 자르거나 [PAD](0)로 채운 shape (문장 수, max_len) 텐서를 돌려줍니다."""
    rows = []
    for text in texts:
        ids = [token_to_id["[CLS]"]] + encode(text, token_to_id)
        ids = ids[:max_len]  # 길면 뒤를 자릅니다
        ids = ids + [token_to_id["[PAD]"]] * (max_len - len(ids))  # 짧으면 뒤를 [PAD](0)로 채웁니다
        rows.append(ids)
    return torch.tensor(rows)


def count_parameters(model):
    """모델의 파라미터(가중치, 절편, 배치 정규화의 gamma와 beta) 개수를 모두 더해 돌려줍니다."""
    total = 0
    for param in model.parameters():
        total += param.numel()
    return total


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    model.eval()  # 평가 모드: Dropout이 아무것도 0으로 만들지 않습니다
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples


def train_classifier(model, train_loader, valid_loader, epochs):
    """AdamW로 epochs번 학습하며, 에폭마다 학습 손실, 학습 정확도, 검증 손실, 검증 정확도를 기록한 DataFrame을 돌려줍니다."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=0.01)
    rows = []
    for epoch in range(1, epochs + 1):
        # 학습 모드: Dropout이 켜집니다
        model.train()
        total_loss = 0.0
        correct = 0
        for X_batch, y_batch in train_loader:
            # 미니배치를 모델이 있는 계산 장치로 옮깁니다
            X_batch = X_batch.to(device)  # (배치, 토큰 수)
            y_batch = y_batch.to(device)  # (배치,)

            # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
            optimizer.zero_grad()
            logits = model(X_batch)  # (배치, 클래스 수)
            loss = loss_fn(logits, y_batch)
            loss.backward()
            optimizer.step()

            # 이 미니배치의 손실과 맞힌 수를 더합니다
            total_loss += loss.item()
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

        # 학습 손실은 미니배치 손실의 평균, 학습 정확도는 맞힌 수 / 학습 데이터 수
        train_loss = total_loss / len(train_loader)
        train_accuracy = correct / len(train_loader.dataset)

        # 검증 데이터로 평가합니다(evaluate가 평가 모드로 바꿉니다)
        valid_loss, valid_accuracy = evaluate(model, valid_loader, loss_fn)
        print(f"에폭 {epoch}: 학습 손실 {train_loss:.4f}, 학습 정확도 {train_accuracy:.4f} | 검증 손실 {valid_loss:.4f}, 검증 정확도 {valid_accuracy:.4f}")
        rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 정확도": train_accuracy,
                     "검증 손실": valid_loss, "검증 정확도": valid_accuracy})
    return pd.DataFrame(rows).set_index("에폭")

## 1. 문자 단위 토큰과 어휘 사전

**배경**: 교안 3절처럼 문장을 문자 단위로 토큰화하고(문자 단위 토큰화, character-level tokenization), 학습 문장으로만 어휘 사전(vocabulary)을 만듭니다. 학습, 검증 나누기는 제공합니다.

**요구사항**
- (1) `train_table`의 문장에서 문자마다 나온 횟수를 **`char_counts`**(`Counter`)로 셉니다. 공백 문자 `' '`도 토큰 하나입니다.
- (2) 어휘 사전 **`intent_vocab`**(리스트)과 토큰 -> 토큰 ID 딕셔너리 **`intent_token_to_id`** 를 만듭니다. 어휘 사전은 `[PAD]`, `[UNK]`, `[CLS]` 다음에 `min_count`번 이상 나온 문자를 많이 나온 순서로 붙입니다.

**예시**: `창문 좀 닫아`는 문자 단위 토큰 7개(공백 2개 포함)입니다.

**확인 기준**: 어휘 사전이 교안 3-2와 같은 규칙으로 만들어졌는지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 3-2(문자 빈도, 어휘 사전과 토큰 ID) 코드를 이번 데이터 이름으로 다시 씁니다.

세부구현:
1. Counter의 update에 문장의 문자 리스트를 넘깁니다.
2. Counter의 most_common()은 많이 나온 순서로 (토큰, 횟수)를 돌려줍니다.
```

</details>

In [ ]:
# [작성] 학습 문장으로 어휘 사전을 만드세요.
from collections import Counter

# 학습 18,000, 검증 2,000으로 나눕니다(제공)
shuffled = intent_train.sample(frac=1, random_state=0)
train_table = shuffled.iloc[:18000]
valid_table = shuffled.iloc[18000:]

# 1. 학습 문장에서 문자마다 나온 횟수
char_counts = Counter()
...

# 2. 어휘 사전과 토큰 -> 토큰 ID
intent_vocab = ["[PAD]", "[UNK]", "[CLS]"]
...
intent_token_to_id = {}
...
print("어휘 크기:", len(intent_vocab), "| 앞 10개:", intent_vocab[:10])

In [ ]:
# [자가채점] 문자 빈도와 어휘 사전을 불변식으로 확인합니다.
assert isinstance(char_counts, Counter), "char_counts는 Counter입니다. 빈칸을 채웠는지 확인하세요."
assert char_counts[" "] > 0, "문자 단위 토큰화에서는 공백 문자 ' '도 토큰입니다."
assert sum(char_counts.values()) == train_table["문장"].str.len().sum(), "char_counts는 train_table 문장의 문자를 모두 센 것이어야 합니다."

assert intent_vocab[:3] == ["[PAD]", "[UNK]", "[CLS]"], "어휘 사전의 토큰 ID 0, 1, 2는 [PAD], [UNK], [CLS] 순서입니다."
assert len(intent_vocab) == len(set(intent_vocab)), "어휘 사전에 같은 토큰이 두 번 들어 있습니다."
for k in range(3, len(intent_vocab)):
    assert char_counts[intent_vocab[k]] >= min_count, f"'{intent_vocab[k]}'은 학습 문장에서 {min_count}번 미만 나온 문자입니다."
    if k > 3:
        assert char_counts[intent_vocab[k - 1]] >= char_counts[intent_vocab[k]], "어휘 사전의 문자는 많이 나온 순서여야 합니다."
for token in char_counts:
    if char_counts[token] >= min_count:
        assert token in intent_token_to_id, f"'{token}'은 {min_count}번 이상 나왔는데 어휘 사전에 없습니다."
for check_id in [0, 1, 2, 3, len(intent_vocab) - 1]:
    assert intent_token_to_id[intent_vocab[check_id]] == check_id, "intent_token_to_id[토큰]이 intent_vocab 리스트의 인덱스(토큰 ID)와 다릅니다."
assert len(intent_token_to_id) == len(intent_vocab), "intent_token_to_id에는 어휘 사전의 토큰이 모두 있어야 합니다."
print("✅ 1번 통과")

**결과 읽기**
- 어휘 크기를 적고, 영화 리뷰의 어휘 사전(교안 3-2)과 비교합니다.

## 2. max_len과 텐서

**배경**: 교안 4절처럼 [CLS]를 붙이고 자르거나 패딩해 같은 시퀀스 길이로 맞춥니다. 의도 문장은 영화 리뷰보다 짧아 64보다 작은 `max_len`으로도 대부분 들어갑니다.

**요구사항**
- (1) `train_table` 문장의 글자 수 **`train_lengths`**(Series)와 그 95% 지점 **`length_95`** 를 계산합니다.
- (2) `intent_max_len`(정수)을 40으로 정합니다. 95% 지점에 [CLS] 토큰 1개를 더한 길이보다 넉넉한 값입니다.
- (3) `make_tensor`로 학습, 검증, 테스트 문장을 **`X_intent_train`**, **`X_intent_valid`**, **`X_intent_test`** 로, 의도를 정수 텐서 **`y_intent_train`**, **`y_intent_valid`**, **`y_intent_test`** 로 만듭니다.

**예시**: 95% 지점이 30글자면 [CLS]를 더해 토큰 31개가 들어갈 길이가 필요하므로, 그보다 조금 큰 값을 `max_len`으로 고릅니다.

**확인 기준**: 잘리는 학습 문장이 5% 이하인지와 텐서 shape을 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 2-2의 길이 요약과 4절의 make_tensor, 전체 텐서 만들기를 이번 데이터 이름으로 씁니다.

세부구현:
1. 문장 열의 str.len()은 문장마다 글자 수입니다. 95% 지점은 Series의 quantile입니다.
2. make_tensor는 문장 리스트, 토큰 -> 토큰 ID 딕셔너리, 시퀀스 길이를 받습니다. DataFrame의 열은 tolist()로 리스트가 됩니다.
3. 의도 열의 values를 torch.tensor에 넣으면 정수 텐서가 됩니다.
```

</details>

In [ ]:
# [작성] 문장 길이로 max_len을 정하고, 문장과 의도를 텐서로 바꾸세요.

# 1. 학습 문장의 글자 수와 95% 지점
train_lengths = ...
length_95 = ...
print("길이 중앙값:", train_lengths.median(), "| 95% 지점:", length_95, "| 가장 긴 문장:", train_lengths.max())

# 2. [CLS]까지 넉넉히 들어가는 길이
intent_max_len = ...

# 3. 문장 -> (문장 수, intent_max_len) 텐서, 의도 -> 정수 텐서
X_intent_train = ...
y_intent_train = ...
X_intent_valid = ...
y_intent_valid = ...
X_intent_test = ...
y_intent_test = ...
print("학습 텐서 shape:", tuple(X_intent_train.shape), "| 검증 텐서 shape:", tuple(X_intent_valid.shape), "| 테스트 텐서 shape:", tuple(X_intent_test.shape))

In [ ]:
# [자가채점] max_len, 잘리는 비율, 텐서 shape과 [CLS] 위치를 확인합니다.
assert len(train_lengths) == 18000 and train_lengths.min() > 0, "train_lengths는 train_table 문장 18,000개의 글자 수입니다."
assert 25 <= length_95 <= 40, f"length_95가 {length_95}입니다. 학습 문장 글자 수의 95% 지점(quantile 0.95)을 계산했는지 확인하세요."
assert isinstance(intent_max_len, int), "intent_max_len은 파이썬 정수입니다. 계산한 값이면 int(...)로 바꾸세요."
assert length_95 + 1 <= intent_max_len <= 48, f"intent_max_len이 {intent_max_len}입니다. [CLS]를 더한 길이(length_95 + 1)보다 넉넉하게, 이 과제는 40을 씁니다."

# [CLS] 1개를 더한 길이가 intent_max_len을 넘는 학습 문장의 비율
truncated = ((train_lengths + 1) > intent_max_len).mean()
print("잘리는 학습 문장 비율:", round(truncated, 4))
assert truncated <= 0.05, "잘리는 학습 문장이 5%를 넘습니다."

assert X_intent_train.shape == (18000, intent_max_len) and X_intent_valid.shape == (2000, intent_max_len) and X_intent_test.shape == (3000, intent_max_len), "세 텐서는 (문장 수, intent_max_len)입니다. make_tensor에 intent_token_to_id와 intent_max_len을 넘겼는지 확인하세요."
assert (X_intent_train[:, 0] == 2).all() and (X_intent_valid[:, 0] == 2).all() and (X_intent_test[:, 0] == 2).all(), "모든 문장의 0번 위치는 [CLS](토큰 ID 2)입니다."
assert int(X_intent_train.max()) < len(intent_vocab), "텐서에 어휘 크기 이상인 토큰 ID가 있습니다."
assert y_intent_train.shape == (18000,) and y_intent_valid.shape == (2000,) and y_intent_test.shape == (3000,), "y_intent_*는 문장 수만큼의 정수 텐서입니다."
assert y_intent_train.dtype == torch.long and int(y_intent_train.min()) == 0 and int(y_intent_train.max()) == 6, "의도는 0~6 정수입니다. 의도 열(숫자)로 만들었는지 확인하세요."
print("✅ 2번 통과")

**결과 읽기**
- 95% 지점과 잘리는 문장 비율을 적고, 40이 이 데이터에 충분한 이유를 영화 리뷰의 `max_len`(64)과 비교해 적습니다.

## 3. 토큰 임베딩

**배경**: 교안 5절의 `nn.Embedding`은 토큰 ID로 임베딩 행렬의 행을 꺼냅니다. 같은 문자가 두 번 나오는 문장으로 확인하고, 임베딩 차원은 교안(64)보다 작은 32로 합니다.

**요구사항**
- (1) `torch.manual_seed(0)` 직후 임베딩 행렬이 (어휘 크기, 32)이고 `[PAD]`(토큰 ID 0) 행을 0으로 두는 **`intent_embedding`** 을 만듭니다.
- (2) `make_tensor`로 예시 문장 `example_sentence`(제공)를 (1, `intent_max_len`) 텐서 **`example_ids`** 로 바꿉니다.
- (3) `example_ids`의 토큰 벡터 **`example_vectors`**(1, `intent_max_len`, 32)를 꺼냅니다.
- (4) 2번 위치와 7번 위치(두 `일`)의 토큰 벡터가 같은지 **`same_char_equal`**(True/False)로 확인합니다.

**예시**: `내일 오후 일정 알려줘`는 [CLS]가 0번 위치라서 `내`가 1번, 첫 `일`이 2번, 두 번째 `일`이 7번 위치입니다.

**확인 기준**: 위치마다 벡터가 임베딩 행렬에서 그 토큰 ID의 행과 같은지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 5절의 임베딩 행렬 만들기와 행 꺼내기를 이번 크기로 씁니다.

세부구현:
1. nn.Embedding의 두 인자는 행 수(어휘 크기)와 임베딩 차원이고, 0으로 둘 행은 padding_idx 인자로 정합니다.
2. 행 꺼내기는 nn.Embedding 층을 함수처럼 부르며 토큰 ID 텐서를 넣습니다.
3. 두 벡터가 완전히 같은지는 torch.equal로 확인합니다.
```

</details>

In [ ]:
# [작성] 토큰 임베딩 행렬을 만들고 예시 문장의 토큰 벡터를 꺼내세요.

# 예시 문장(제공): 문자 토큰 12개, '일'이 두 번 나옵니다
example_sentence = "내일 오후 일정 알려줘"

# 1. (어휘 크기, 32) 임베딩 행렬
torch.manual_seed(0)
intent_embedding = ...

# 2, 3. 토큰 ID (1, intent_max_len) -> 토큰 벡터 (1, intent_max_len, 32)
example_ids = ...
with torch.no_grad():
    example_vectors = ...
print("토큰 ID shape:", tuple(example_ids.shape), "| 토큰 벡터 shape:", tuple(example_vectors.shape))
print("위치 0~12의 토큰 ID:", example_ids[0, :13].tolist())

# 4. 두 '일'(2번, 7번 위치)의 토큰 벡터가 같은가
same_char_equal = ...
print("두 '일'의 토큰 벡터가 같은가:", same_char_equal)

In [ ]:
# [자가채점] 임베딩 행렬의 설정과 shape, 행 꺼내기, [PAD] 벡터, 같은 문자를 확인합니다.
assert isinstance(intent_embedding, nn.Embedding), "intent_embedding은 nn.Embedding으로 만드세요. 빈칸을 채웠는지 확인하세요."
assert intent_embedding.num_embeddings == len(intent_vocab) and intent_embedding.embedding_dim == 32, f"임베딩 행렬의 shape이 {tuple(intent_embedding.weight.shape)}입니다. (어휘 크기, 32)로 만드세요."
assert intent_embedding.padding_idx == 0, "[PAD](토큰 ID 0) 행을 0으로 두려면 padding_idx를 0으로 줍니다."
assert tuple(example_ids.shape) == (1, intent_max_len) and example_ids[0, 0].item() == 2, "example_ids는 (1, intent_max_len)이고 0번 위치가 [CLS](토큰 ID 2)입니다. make_tensor에 문장 리스트를 넣었는지 확인하세요."
assert example_ids[0, 2].item() == intent_token_to_id["일"] and example_ids[0, 7].item() == intent_token_to_id["일"], "2번, 7번 위치는 '일'의 토큰 ID여야 합니다."
assert tuple(example_vectors.shape) == (1, intent_max_len, 32), f"example_vectors의 shape이 {tuple(example_vectors.shape)}입니다. (1, intent_max_len, 32)여야 합니다."
for position in [0, 1, 2, 7, 12]:
    token_id = example_ids[0, position].item()
    assert torch.equal(example_vectors[0, position], intent_embedding.weight[token_id].detach()), f"{position}번 위치의 벡터가 임베딩 행렬의 {token_id}번 행과 다릅니다. intent_embedding에 example_ids를 넣었는지 확인하세요."
assert torch.all(example_vectors[0, 13:] == 0).item(), "[PAD] 위치(13번부터)의 벡터는 모두 0이어야 합니다. padding_idx를 줬는지 확인하세요."
assert bool(same_char_equal) is True, "same_char_equal이 True가 아닙니다. 2번과 7번 위치의 토큰 벡터를 torch.equal로 비교하세요."
print("✅ 3번 통과")

**결과 읽기**
- `same_char_equal`과 [PAD] 위치의 벡터를 보고, 토큰 임베딩만으로 `내일`의 `일`과 `일정`의 `일`을 구분할 수 있는지 적습니다.

## 4. 어텐션과 위치 인코딩

**배경**: 교안 6-4와 7-3의 순서 실험을 `nn.MultiheadAttention`으로 합니다. [PAD] 없이 진짜 토큰 13개([CLS]와 문자 토큰 12개)만 씁니다.

**요구사항**
- (1) `torch.manual_seed(0)` 직후 `embed_dim=32`, `num_heads=4`이고 (배치, 토큰 수, 임베딩 차원) 순서로 받는 어텐션 층 **`mha`** 를 만듭니다.
- (2) `real_vectors`(제공)로 Self-Attention을 해 출력 **`plain_output`** 과 헤드 평균 어텐션 가중치 **`plain_weights`** 를 받습니다.
- (3) 순서를 뒤집은 `real_vectors[:, reverse_order]`의 출력 **`reversed_output`** 을 받습니다. 이 출력을 `reverse_order`로 다시 뒤집으면 `plain_output`과 같은지 **`order_ignored`**(True/False, `atol=1e-5`)로 확인합니다.
- (4) `positional_encoding`으로 위치 `intent_max_len`개, 32차원인 위치 인코딩 행렬 **`intent_position_table`** 을 만듭니다. 제공 줄이 위치 0~12의 위치 벡터를 두 입력에 똑같이 더합니다. 두 입력의 출력 **`position_output`**, **`reversed_position_output`** 을 받습니다.
- (5) (3)과 같은 비교를 **`order_ignored_with_position`** 으로 확인합니다.

**예시**: 뒤집은 입력은 0번 위치가 `줘`, 12번 위치가 [CLS]입니다. 출력도 뒤집힌 순서로 나오므로 다시 뒤집어 비교합니다.

**확인 기준**: `order_ignored`가 True이고 `order_ignored_with_position`이 False인지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 6-4, 7-3 실험의 어텐션 계산 대신 교안 8-1의 nn.MultiheadAttention을 씁니다.

세부구현:
1. 배치가 맨 앞인 입력을 받으려면 batch_first를 켭니다.
2. Self-Attention은 query, key, value 인자에 같은 텐서를 넣고, 층은 (출력, 어텐션 가중치)를 돌려줍니다.
3. 두 텐서가 허용 오차 안에서 같은지는 torch.allclose로 확인합니다.
4. 위치 인코딩 행렬은 교안 7-2처럼 positional_encoding에 위치 수와 차원을 넘겨 만듭니다.
```

</details>

In [ ]:
# [작성] 어텐션 층으로 순서를 뒤집은 입력을 비교하고, 위치 인코딩을 더해 다시 비교하세요.

# 진짜 토큰 13개([CLS]와 문자 토큰 12개)의 토큰 벡터와 뒤집는 순서(제공)
real_vectors = example_vectors[:, :13]  # (1, 13, 32)
reverse_order = list(range(12, -1, -1))  # 12부터 0까지 거꾸로
reversed_input = real_vectors[:, reverse_order]  # 0번 위치가 '줘', 12번 위치가 [CLS]

# 1. 어텐션 층
torch.manual_seed(0)
mha = ...

with torch.no_grad():
    # 2. 원래 순서
    plain_output, plain_weights = ...

    # 3. 뒤집은 순서
    reversed_output, _ = ...
order_ignored = ...
print("출력 shape:", tuple(plain_output.shape), "| 어텐션 가중치 shape:", tuple(plain_weights.shape))
print("위치 인코딩 없이, 뒤집은 출력을 되돌리면 같은가:", order_ignored)

# 4. 위치 인코딩 행렬 (intent_max_len, 32)와, 위치 벡터를 더한 두 입력
intent_position_table = ...
with torch.no_grad():
    # 뒤집지 않은 같은 위치 벡터(위치 0~12)를 두 입력에 더합니다(제공)
    positions = intent_position_table[:13]  # (13, 32)
    position_input = real_vectors + positions
    reversed_position_input = reversed_input + positions
    position_output, _ = ...
    reversed_position_output, _ = ...

# 5. 같은 비교
order_ignored_with_position = ...
print("위치 인코딩을 더하고, 뒤집은 출력을 되돌리면 같은가:", order_ignored_with_position)

In [ ]:
# [자가채점] 층의 설정, shape, 어텐션 가중치 합, 위치 인코딩 행렬, 두 비교 결과를 확인합니다.
assert isinstance(mha, nn.MultiheadAttention), "mha는 nn.MultiheadAttention으로 만드세요."
assert mha.embed_dim == 32 and mha.num_heads == 4, f"mha의 embed_dim, num_heads가 {mha.embed_dim}, {mha.num_heads}입니다. 32와 4로 만드세요."
assert mha.batch_first is True, "입력이 (배치, 토큰 수, 임베딩 차원) 순서이므로 batch_first를 켜야 합니다. 꺼 두면 에러 없이 축을 잘못 읽습니다."
assert tuple(plain_output.shape) == (1, 13, 32), f"plain_output의 shape이 {tuple(plain_output.shape)}입니다. real_vectors를 query, key, value에 넣었는지 확인하세요."
assert tuple(plain_weights.shape) == (1, 13, 13), f"plain_weights의 shape이 {tuple(plain_weights.shape)}입니다. 헤드 평균 어텐션 가중치는 (배치, Query 토큰 수, Key 토큰 수)입니다."
assert torch.allclose(plain_weights.sum(dim=-1), torch.ones(1, 13), atol=1e-5), "Query 하나의 어텐션 가중치는 Key에 대해 합이 1이어야 합니다."
assert tuple(reversed_output.shape) == (1, 13, 32), "reversed_output은 뒤집은 입력 (1, 13, 32)의 출력입니다."
assert not torch.allclose(reversed_output, plain_output, atol=1e-5), "reversed_output이 plain_output과 순서까지 같습니다. 뒤집은 입력을 넣었는지 확인하세요."
assert bool(order_ignored) is True, "위치 인코딩이 없으면 뒤집은 출력을 되돌린 값이 plain_output과 같아야 합니다. reversed_output을 reverse_order로 다시 뒤집어 비교했는지 확인하세요."

assert isinstance(intent_position_table, torch.Tensor), "intent_position_table은 positional_encoding이 돌려준 텐서입니다. nn.Embedding이 아니라 positional_encoding으로 만드세요."
assert tuple(intent_position_table.shape) == (intent_max_len, 32), f"위치 인코딩 행렬의 shape이 {tuple(intent_position_table.shape)}입니다. (intent_max_len, 32)로 만드세요."
assert intent_position_table.requires_grad is False, "위치 인코딩 행렬은 학습되지 않는 고정 값입니다."
assert torch.equal(intent_position_table, positional_encoding(intent_max_len, 32)), "intent_position_table이 sin, cos 위치 인코딩 행렬과 다릅니다. positional_encoding에 위치 수와 차원을 넘겼는지 확인하세요."
assert tuple(position_output.shape) == (1, 13, 32) and tuple(reversed_position_output.shape) == (1, 13, 32), "위치 벡터를 더한 두 출력은 (1, 13, 32)입니다."
assert not torch.allclose(position_output, plain_output, atol=1e-5), "position_output이 plain_output과 같습니다. 입력에 위치 벡터를 더했는지 확인하세요."
assert bool(order_ignored_with_position) is False, "위치 벡터를 더하면 뒤집은 출력을 되돌려도 원래 출력과 달라야 합니다. position_input과 reversed_position_input을 각각 넣었는지 확인하세요."
print("✅ 4번 통과")

**결과 읽기**
- 두 비교 결과를 적고, 위치 인코딩이 없을 때 `내일 오후 일정 알려줘`와 문자 순서를 거꾸로 놓은 문장을 어텐션이 어떻게 보는지 적습니다.

## 5. 패딩 마스크

**배경**: 교안 8-1, 8-2처럼 예시 문장 텐서(진짜 토큰 13개, [PAD] 27개)를 4번의 `mha`에 넣고, 0번 위치([CLS])의 Query가 [PAD] 위치에 준 어텐션 가중치를 봅니다.

**요구사항**
- (1) 3번의 토큰 벡터 `example_vectors`에 4번의 위치 인코딩 행렬 `intent_position_table`을 더해 입력 벡터 **`example_input`**(1, `intent_max_len`, 32)을 만듭니다.
- (2) 마스크 없이 `example_input`으로 Self-Attention을 해 헤드 평균 어텐션 가중치 **`weights_without_mask`** 를 받습니다. 0번 행([CLS] Query)이 [PAD] 위치(13번부터)에 준 어텐션 가중치의 합 **`pad_weight_without_mask`**(실수)를 계산합니다.
- (3) `example_ids`로 [PAD] 위치가 True인 패딩 마스크 **`example_mask`** 를 만듭니다.
- (4) 마스크를 `key_padding_mask`로 넘겨 다시 계산한 어텐션 가중치 **`weights_with_mask`** 와, (2)와 같은 합 **`pad_weight_with_mask`** 를 구합니다.

**예시**: 어텐션 가중치의 0번 행은 [CLS]가 위치 40개에 나눠 준 어텐션 가중치이고 합이 1입니다. 그 가운데 13번부터 39번까지가 [PAD] 위치입니다.

**확인 기준**: 마스크 없이는 [PAD] 위치에 어텐션 가중치가 많이 가고, 마스크와 함께는 0인지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 8-1, 8-2의 두 셀(마스크 없이, 마스크와 함께)을 이번 이름으로 씁니다.

세부구현:
1. example_vectors는 (1, 40, 32), 위치 인코딩 행렬은 (40, 32)라서 그대로 더할 수 있습니다.
2. 어텐션 가중치에서 0번 행의 13번 열부터 끝까지를 골라 sum()으로 더하고, item()으로 실수로 바꿉니다.
3. 패딩 마스크는 토큰 ID 텐서를 [PAD]의 토큰 ID(0)와 비교해 만듭니다.
4. 마스크는 어텐션 층을 부를 때 key_padding_mask 인자로 넘깁니다.
```

</details>

In [ ]:
# [작성] 예시 문장으로 [CLS]가 [PAD] 위치에 준 어텐션 가중치를 마스크 없이, 마스크와 함께 비교하세요.

# 1. 입력 벡터 = 토큰 벡터 + 위치 인코딩
example_input = ...

# 2. 마스크 없이: 헤드 평균 어텐션 가중치 (1, intent_max_len, intent_max_len)
with torch.no_grad():
    _, weights_without_mask = ...
pad_weight_without_mask = ...
print("입력 shape:", tuple(example_input.shape), "| 어텐션 가중치 shape:", tuple(weights_without_mask.shape))
print("마스크 없이 [CLS]가 [PAD] 위치에 준 어텐션 가중치 합:", round(pad_weight_without_mask, 4))

# 3. 패딩 마스크: [PAD](토큰 ID 0) 위치가 True
example_mask = ...
print("마스크의 True 수:", example_mask.sum().item())

# 4. 마스크와 함께
with torch.no_grad():
    _, weights_with_mask = ...
pad_weight_with_mask = ...
print("마스크와 함께 [CLS]가 [PAD] 위치에 준 어텐션 가중치 합:", round(pad_weight_with_mask, 4))

In [ ]:
# [자가채점] 입력, 두 어텐션 가중치, 마스크, [PAD] 위치의 어텐션 가중치 합을 확인합니다.
assert isinstance(example_input, torch.Tensor) and tuple(example_input.shape) == (1, intent_max_len, 32), "example_input은 (1, intent_max_len, 32)입니다."
assert not torch.allclose(example_input, example_vectors), "example_input이 토큰 벡터 그대로입니다. 위치 인코딩 행렬을 더했는지 확인하세요."
assert torch.allclose(example_input[0, 13:], intent_position_table[13:], atol=1e-6), "[PAD] 위치는 토큰 벡터가 0이라 위치 벡터만 남아야 합니다. 위치 0부터 위치 인코딩 행렬 전체를 더했는지 확인하세요."

assert tuple(weights_without_mask.shape) == (1, intent_max_len, intent_max_len), "weights_without_mask는 (1, intent_max_len, intent_max_len) 어텐션 가중치입니다. mha를 교안 8-1처럼 기본 호출로 부르세요."
assert torch.allclose(weights_without_mask.sum(dim=-1), torch.ones(1, intent_max_len), atol=1e-5), "Query 하나의 어텐션 가중치는 Key에 대해 합이 1이어야 합니다."
assert isinstance(pad_weight_without_mask, float), "pad_weight_without_mask는 파이썬 실수입니다. 합계 텐서에 item()을 붙이세요."
assert 0.3 < pad_weight_without_mask < 0.95, f"pad_weight_without_mask가 {pad_weight_without_mask:.3f}입니다. 0번 행에서 [PAD] 위치(13번부터 끝까지)만 더했는지 확인하세요."

assert example_mask.dtype == torch.bool and tuple(example_mask.shape) == (1, intent_max_len), "example_mask는 shape이 (1, intent_max_len)인 참/거짓(bool) 텐서입니다. example_ids를 0과 비교해 만드세요."
assert example_mask.sum().item() == intent_max_len - 13, "마스크의 True 수는 [PAD] 수(진짜 토큰 13개를 뺀 위치 수)와 같아야 합니다."

assert tuple(weights_with_mask.shape) == (1, intent_max_len, intent_max_len), "weights_with_mask는 헤드 평균 어텐션 가중치 (1, intent_max_len, intent_max_len)입니다."
assert weights_with_mask[0, :, 13:].abs().max().item() < 1e-6, "마스크를 넘긴 어텐션 가중치는 [PAD] 위치(13번 열부터)에서 0이어야 합니다. key_padding_mask에 example_mask를 넘겼는지 확인하세요."
assert isinstance(pad_weight_with_mask, float) and abs(pad_weight_with_mask) < 1e-6, "마스크와 함께라면 [CLS]가 [PAD] 위치에 준 어텐션 가중치 합은 0입니다. (2)와 같은 위치를 더했는지 확인하세요."
print("✅ 5번 통과")

**결과 읽기**
- 마스크 없이와 마스크와 함께의 [PAD] 위치 어텐션 가중치 합을 적고, 패딩한 텐서를 어텐션에 넣을 때 마스크가 필요한 이유를 한 문장으로 적습니다.

## 6. 인코더 층 쌓기

**배경**: 교안 9-3의 `nn.TransformerEncoderLayer`와 `nn.TransformerEncoder`를 이번 크기로 만들고, 층을 쌓을 때 파라미터 수와 출력 shape을 확인합니다.

**요구사항**
- (1) `d_model=32`, `nhead=4`, `dim_feedforward=128`이고 (배치, 토큰 수, d_model) 순서로 받는 인코더 층 **`encoder_layer_32`** 를 만듭니다.
- (2) 이 층을 3개 쌓은 인코더 **`encoder_32`** 를 만듭니다. 교안 9-3과 같게 `enable_nested_tensor=False`를 줍니다(`encoder_models.py`의 `TransformerClassifier`와 같은 설정입니다).
- (3) `count_parameters`로 층 하나와 인코더의 파라미터 수 **`layer_params`**, **`encoder_params`** 를 셉니다.
- (4) 평가 모드의 `encoder_32`에 5번의 `example_input`과 패딩 마스크 `example_mask`를 넣은 출력 **`encoded_example`** 을 받습니다.

**예시**: 교안의 인코더 층(d_model 64, FFN 가운데 256)은 층 하나 49,984개, 2층 99,968개였습니다.

**확인 기준**: 층 하나의 파라미터 수와, 인코더가 그 정확히 3배인지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 9-3 셀의 인자를 이번 크기로 바꿉니다.

세부구현:
1. 인코더 층에도 batch_first를 켭니다. FFN 가운데 크기는 dim_feedforward 인자입니다.
2. 쌓을 층 수는 num_layers 인자입니다.
3. nn.TransformerEncoder에 넘기는 패딩 마스크의 인자 이름은 src_key_padding_mask입니다(교안 10-1).
4. 출력은 Dropout을 끈 평가 모드에서 기울기 없이 받습니다.
```

</details>

In [ ]:
# [작성] 인코더 층을 3개 쌓고 파라미터 수와 출력 shape을 확인하세요.

# 1, 2. 인코더 층 하나와 3층 인코더
encoder_layer_32 = ...
encoder_32 = ...

# 3. 파라미터 수
layer_params = ...
encoder_params = ...
print("층 하나:", layer_params, "| 인코더(3층):", encoder_params)

# 4. 패딩 마스크와 함께 넣은 출력
encoder_32.eval()  # Dropout을 끕니다
with torch.no_grad():
    encoded_example = ...
print("입력 shape:", tuple(example_input.shape), "| 출력 shape:", tuple(encoded_example.shape))

In [ ]:
# [자가채점] 층의 설정, 층 수, 파라미터 수, 출력 shape을 확인합니다.
assert isinstance(encoder_layer_32, nn.TransformerEncoderLayer), "encoder_layer_32는 nn.TransformerEncoderLayer로 만드세요. 빈칸을 채웠는지 확인하세요."
assert encoder_layer_32.self_attn.embed_dim == 32 and encoder_layer_32.self_attn.num_heads == 4, "encoder_layer_32는 d_model 32, nhead 4입니다."
assert encoder_layer_32.linear1.out_features == 128, "encoder_layer_32의 dim_feedforward는 128입니다."
assert encoder_layer_32.self_attn.batch_first is True, "encoder_layer_32에 batch_first를 켜야 (배치, 토큰 수, d_model) 순서로 읽습니다."
assert isinstance(encoder_32, nn.TransformerEncoder) and len(encoder_32.layers) == 3, "encoder_32는 층 3개를 쌓은 nn.TransformerEncoder입니다. num_layers를 확인하세요."
assert layer_params == 12704, f"layer_params가 {layer_params}입니다. count_parameters에 encoder_layer_32를 넣었는지, 층의 설정이 지문과 같은지 확인하세요."
assert encoder_params == 3 * layer_params, f"encoder_params({encoder_params})가 층 하나의 3배가 아닙니다. count_parameters에 encoder_32를 넣었는지 확인하세요."
assert tuple(encoded_example.shape) == tuple(example_input.shape), f"encoded_example의 shape이 {tuple(encoded_example.shape)}입니다. 인코더의 출력은 입력과 같은 shape이어야 합니다."
print("✅ 6번 통과")

**결과 읽기**
- 층 하나와 3층의 파라미터 수, 출력 shape을 적고, 같은 층을 여러 개 쌓을 수 있는 이유를 적습니다.

## 7. 의도 분류 모델과 학습 전 결과

**배경**: 교안 10절의 `TransformerClassifier`를 클래스 7개로 만듭니다. 학습 전 손실과 정확도, 확인 문장 5개의 예측을 먼저 남겨 학습 뒤(9번)와 비교합니다.

**요구사항**
- (1) `torch.manual_seed(0)` 직후 어휘 크기 `len(intent_vocab)`, 클래스 7개, `max_len=intent_max_len`인 **`intent_model`** 을 만들어 계산 장치에 올립니다. 나머지 크기는 기본값입니다.
- (2) 제공 DataLoader로 학습 전 검증 손실과 정확도 **`valid_loss_before`**, **`valid_accuracy_before`** 를 `evaluate`로 계산합니다.
- (3) 확인 문장 `check_sentences`(제공)의 학습 전 예측 의도 이름 목록 **`before_names`** 를 만들고, 열이 `문장`, `학습 전 예측`인 DataFrame **`check_table`** 에 담습니다.

**예시**: 클래스 7개에 같은 확률을 주는 모델의 손실은 $-\log(1/7) \approx 1.95$입니다.

**확인 기준**: 학습 전 검증 손실이 1.5보다 크고 정확도가 0.45보다 낮은지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 10-1, 10-2의 코드에서 클래스 수, max_len, 데이터 이름만 바꿉니다.

세부구현:
1. TransformerClassifier는 어휘 크기, 클래스 수(num_classes), max_len을 인자로 받습니다.
2. evaluate는 (손실, 정확도)를 돌려줍니다.
3. 확인 문장 텐서는 2번의 make_tensor로 만들고 계산 장치로 올립니다(.to(device)).
4. 예측은 평가 모드에서 기울기 없이 그 텐서를 모델에 넣고, 로짓이 가장 큰 클래스의 인덱스(argmax)를 고릅니다.
5. 클래스 인덱스는 intent_names의 순서와 같습니다.
```

</details>

In [ ]:
# [작성] 의도 클래스 7개의 분류 모델을 만들고 학습 전 결과를 남기세요.

# DataLoader(제공): 미니배치 크기 128, 학습용만 에폭마다 섞습니다
intent_train_loader = DataLoader(TensorDataset(X_intent_train, y_intent_train), batch_size=128, shuffle=True)
intent_valid_loader = DataLoader(TensorDataset(X_intent_valid, y_intent_valid), batch_size=128, shuffle=False)
intent_test_loader = DataLoader(TensorDataset(X_intent_test, y_intent_test), batch_size=128, shuffle=False)

# 확인 문장(제공): 끝만 다른 문장 셋과, 질문 꼴이 아닌 질문, 수사 의문
check_sentences = ["창문 닫았니", "창문 좀 닫아", "창문 닫았어", "내일 오후 일정 알려줘", "그걸 누가 몰라"]

# 1. 모델
torch.manual_seed(0)
intent_model = ...

# 2. 학습 전 검증 손실과 정확도
valid_loss_before, valid_accuracy_before = ...
print("학습 전 검증 손실:", round(valid_loss_before, 4), "| 검증 정확도:", round(valid_accuracy_before, 4))

# 3. 확인 문장의 학습 전 예측
intent_model.eval()
with torch.no_grad():
    check_logits = ...
before_names = []
...
check_table = ...
display(check_table)

In [ ]:
# [자가채점] 모델의 설정, 학습 전 손실과 정확도, 확인 문장 DataFrame을 확인합니다.
assert isinstance(intent_model, TransformerClassifier), "intent_model은 TransformerClassifier로 만드세요. 빈칸을 채웠는지 확인하세요."
assert intent_model.classifier.out_features == 7, "intent_model의 클래스 수가 7이 아닙니다. num_classes를 줬는지 확인하세요."
assert intent_model.token_embedding.num_embeddings == len(intent_vocab), "intent_model의 어휘 크기가 len(intent_vocab)과 다릅니다."
assert intent_model.position_table.shape[0] == intent_max_len, "intent_model의 max_len이 intent_max_len과 다릅니다. max_len 인자를 줬는지 확인하세요."
assert next(intent_model.parameters()).device.type == device.type, "intent_model을 계산 장치(device)에 올리세요."
assert valid_loss_before > 1.5, f"학습 전 검증 손실이 {valid_loss_before:.3f}입니다. 학습하기 전 모델을 검증 DataLoader로 평가했는지 확인하세요."
assert valid_accuracy_before < 0.45, f"학습 전 검증 정확도가 {valid_accuracy_before:.3f}입니다. 학습하기 전에 evaluate를 불렀는지 확인하세요."
assert len(before_names) == 5, "before_names는 확인 문장 5개의 예측 의도 이름입니다."
assert isinstance(check_table, pd.DataFrame) and list(check_table.columns) == ["문장", "학습 전 예측"], "check_table의 열은 '문장', '학습 전 예측'입니다."
assert check_table["문장"].tolist() == check_sentences, "check_table의 문장 열은 check_sentences 순서 그대로입니다."
for name in check_table["학습 전 예측"]:
    assert name in intent_names, f"'{name}'은 의도 이름이 아닙니다. 클래스 인덱스를 intent_names로 바꿨는지 확인하세요."
print("✅ 7번 통과")

**결과 읽기**
- 학습 전 검증 손실을 $-\log(1/7) \approx 1.95$와, 검증 정확도를 가장 많은 의도(평서)만 답할 때의 정확도(약 0.31)와 비교합니다.

## 8. 학습과 학습 곡선

**배경**: 교안 10-3과 같은 `train_classifier`(AdamW 0.0005, 미니배치 크기 128)로 8에폭 학습하고, 에폭마다 학습과 검증의 손실, 정확도를 그래프로 봅니다.

**요구사항**
- (1) `torch.manual_seed(0)` 직후 `train_classifier`로 `intent_model`을 8에폭 학습한 기록 **`intent_history`** 를 만듭니다.
- (2) `plt.subplots(1, 2, ...)`로 만든 **`axes`** 의 왼쪽에 손실, 오른쪽에 정확도를 그립니다. 서브플롯마다 학습과 검증 두 선을 겹쳐 그리고 범례를 답니다.

**예시**: 기록 DataFrame의 한 행은 `에폭 1: 학습 손실, 학습 정확도, 검증 손실, 검증 정확도`이고, 인덱스는 에폭 번호 1~8입니다.

**확인 기준**: 마지막 검증 정확도가 0.65를 넘는지 확인합니다(가장 많은 의도만 답하면 약 0.31).

<details><summary>힌트</summary>

```text
접근방법:
- 교안 10-3의 학습 셀과 그래프 셀을 이번 이름으로 씁니다.

세부구현:
1. train_classifier는 모델, 학습 DataLoader, 검증 DataLoader, 에폭 수를 받아 기록 DataFrame을 돌려줍니다.
2. 서브플롯 하나에 plot을 두 번 부르고, label을 주면 legend로 범례를 답니다.
```

</details>

In [ ]:
# [작성] (1) 의도 분류 모델을 8에폭 학습하세요.
torch.manual_seed(0)
intent_history = ...
display(intent_history.round(4))

In [ ]:
# [작성] (2) 왼쪽은 손실, 오른쪽은 정확도. 학습과 검증을 한 그래프에 겹쳐 그리세요.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: 학습 손실과 검증 손실
...

# 오른쪽: 학습 정확도와 검증 정확도
...

plt.tight_layout()
plt.show()

In [ ]:
# [자가채점] 기록 DataFrame과 학습 곡선을 확인합니다.
assert isinstance(intent_history, pd.DataFrame) and len(intent_history) == 8, "intent_history는 train_classifier가 돌려준 8에폭 기록 DataFrame입니다."
assert intent_history.index.tolist() == [1, 2, 3, 4, 5, 6, 7, 8], "intent_history의 인덱스는 에폭 번호 1~8입니다."
assert list(intent_history.columns) == ["학습 손실", "학습 정확도", "검증 손실", "검증 정확도"], "intent_history의 열은 학습 손실, 학습 정확도, 검증 손실, 검증 정확도입니다."
assert intent_history["학습 손실"].iloc[-1] < intent_history["학습 손실"].iloc[0], "마지막 에폭의 학습 손실이 첫 에폭보다 작아야 합니다. 7번에서 만든 intent_model을 학습했는지 확인하세요."
assert intent_history["검증 정확도"].iloc[-1] > 0.65, f"마지막 검증 정확도가 {intent_history['검증 정확도'].iloc[-1]:.3f}입니다. 0.65보다 높아야 합니다. 학습 DataLoader와 검증 DataLoader를 바꿔 넣지 않았는지 확인하세요."
assert len(axes) == 2 and len(axes[0].lines) == 2 and len(axes[1].lines) == 2, "왼쪽과 오른쪽 서브플롯에 학습과 검증 선을 하나씩, 서브플롯마다 두 개 그리세요."
assert axes[0].get_legend() is not None and axes[1].get_legend() is not None, "두 서브플롯 모두 범례를 다세요."
print("✅ 8번 통과")

**결과 읽기**
- 첫 에폭과 마지막 에폭의 학습 손실, 검증 정확도를 적습니다.
- 검증 손실이 가장 낮은 에폭과 마지막 에폭을 비교하고, 학습 정확도와 검증 정확도의 차이가 벌어지기 시작하는 에폭이 있는지 적습니다.

## 9. 테스트 평가와 학습 전·후 비교

**배경**: 의도마다 문장 수가 크게 다릅니다(학습 문장 평서 5,558개, 수사 명령 229개). 전체 정확도만 보면 적은 의도의 성능이 가려지므로 의도별 정확도와 혼동 행렬을 함께 봅니다.

PyTorch 신경망 단원 교안 01 7-3에서 반복문으로 만든 혼동 행렬을 `confusion_matrix(정답, 예측)`이 같은 shape으로 돌려줍니다. 9-4의 클래스별 맞힌 비율은 `recall_score(정답, 예측, average=None)`입니다.

**요구사항**
- (1) 테스트 정확도 **`intent_test_accuracy`** 를 7번에서 만든 `intent_test_loader`와 `evaluate`로 계산합니다.
- (2) 테스트 문장 3,000개의 예측 클래스 인덱스 **`intent_predictions`**(정수 텐서, CPU)를 만듭니다.
- (3) `confusion_matrix`로 행은 정답, 열은 예측인 혼동 행렬을 만들고, 인덱스와 열 이름을 `intent_names`로 붙인 DataFrame **`confusion_table`** 을 만듭니다.
- (4) `recall_score`(`average=None`)로 의도별 정확도(그 의도 문장 가운데 맞힌 비율, 재현율)를 계산해 인덱스가 `intent_names`인 Series **`class_accuracy`** 를 만듭니다.
- (5) 확인 문장의 학습 후 예측 의도 이름 목록 **`after_names`** 를 만들고, `check_table`에 열 `학습 후 예측`으로 더합니다.

**예시**: 질문 문장 100개 가운데 90개를 질문으로 맞혔다면 질문의 의도별 정확도는 0.9입니다.

**확인 기준**: 테스트 정확도가 0.70을 넘는지와, 혼동 행렬의 행 합이 의도별 테스트 문장 수인지 확인합니다.

<details><summary>힌트</summary>

```text
접근방법:
- 테스트 평가는 교안 10-4, 예측은 7번 (3)과 같은 방법입니다.
- 혼동 행렬과 재현율은 sklearn.metrics의 두 함수로 계산합니다.

세부구현:
1. 예측은 평가 모드에서 기울기 없이 X_intent_test 전체를 넣고 argmax 한 뒤 CPU로 옮깁니다.
2. confusion_matrix와 recall_score는 정답, 예측 순서로 받습니다.
3. DataFrame은 pd.DataFrame에 index와 columns를, Series는 pd.Series에 index를 줍니다.
```

</details>

In [ ]:
# [작성] 테스트 정확도, 혼동 행렬, 의도별 정확도를 계산하고 같은 문장의 학습 전후 예측을 비교하세요.
from sklearn.metrics import confusion_matrix, recall_score

# 1. 테스트 정확도
_, intent_test_accuracy = ...
print("학습 전 검증 정확도:", round(valid_accuracy_before, 4), "| 테스트 정확도:", round(intent_test_accuracy, 4))

# 2. 테스트 문장 전체의 예측 클래스 인덱스
intent_model.eval()
with torch.no_grad():
    intent_predictions = ...

# 3. 혼동 행렬: 행은 정답, 열은 예측
confusion_table = ...
display(confusion_table)

# 4. 의도별 정확도(재현율)
class_accuracy = ...
display(class_accuracy.round(3))

# 5. 같은 확인 문장의 학습 후 예측
with torch.no_grad():
    check_logits = ...
after_names = []
...
check_table["학습 후 예측"] = after_names
display(check_table)

In [ ]:
# [자가채점] 테스트 정확도, 예측, 혼동 행렬, 의도별 정확도, 학습 후 예측을 확인합니다.
assert intent_test_accuracy > 0.70, f"테스트 정확도가 {intent_test_accuracy:.3f}입니다. 0.70보다 높아야 합니다. 학습한 모델을 테스트 DataLoader로 평가했는지 확인하세요."
assert isinstance(intent_predictions, torch.Tensor) and tuple(intent_predictions.shape) == (3000,), "intent_predictions는 테스트 문장마다 예측 클래스 인덱스 하나, shape (3000,)인 텐서입니다. argmax의 dim을 확인하세요."
assert intent_predictions.device.type == "cpu", "intent_predictions는 CPU 텐서여야 합니다."
assert int(intent_predictions.min()) >= 0 and int(intent_predictions.max()) <= 6, "intent_predictions는 의도 클래스 인덱스(0~6)입니다. 로짓이 아니라 로짓이 가장 큰 클래스의 인덱스를 담으세요."
predictions_accuracy = (intent_predictions == y_intent_test).float().mean().item()
assert abs(predictions_accuracy - intent_test_accuracy) < 0.005, "intent_predictions로 계산한 정확도가 intent_test_accuracy와 다릅니다. 같은 학습한 모델로 예측했는지 확인하세요."

true_counts = torch.bincount(y_intent_test, minlength=7).tolist()
assert isinstance(confusion_table, pd.DataFrame) and confusion_table.shape == (7, 7), "confusion_table은 shape (7, 7)인 DataFrame입니다."
assert list(confusion_table.index) == intent_names and list(confusion_table.columns) == intent_names, "confusion_table의 인덱스와 열 이름은 intent_names입니다."
assert confusion_table.sum(axis=1).tolist() == true_counts, "행마다 합은 그 의도의 테스트 문장 수여야 합니다. confusion_matrix에 정답, 예측 순서로 넣었는지 확인하세요."
assert abs(confusion_table.values.trace() / 3000 - intent_test_accuracy) < 0.005, "혼동 행렬의 대각선 합(맞힌 수)이 테스트 정확도와 맞지 않습니다."

assert isinstance(class_accuracy, pd.Series) and list(class_accuracy.index) == intent_names, "class_accuracy는 인덱스가 intent_names인 Series입니다."
assert class_accuracy.min() >= 0 and class_accuracy.max() <= 1, "의도별 정확도는 0~1입니다."
weighted_accuracy = (class_accuracy.values * true_counts).sum() / 3000
assert abs(weighted_accuracy - intent_test_accuracy) < 0.005, "의도별 정확도를 문장 수로 가중 평균하면 전체 정확도와 같아야 합니다. recall_score에 average=None과 정답, 예측 순서를 줬는지 확인하세요."

assert len(after_names) == 5, "after_names는 확인 문장 5개의 학습 후 예측 의도 이름입니다."
assert list(check_table.columns) == ["문장", "학습 전 예측", "학습 후 예측"], "check_table의 열은 '문장', '학습 전 예측', '학습 후 예측'입니다."
for name in check_table["학습 후 예측"]:
    assert name in intent_names, f"'{name}'은 의도 이름이 아닙니다."
print("✅ 9번 통과")

**결과 읽기**
- 학습 전 검증 정확도와 테스트 정확도를 적고, 의도별 정확도가 높은 의도와 낮은 의도를 학습 문장 수와 함께 적습니다.
- 의도별 정확도가 가장 낮은 의도의 행에서 가장 큰 값(정답 -> 예측)을 찾아, 두 의도가 헷갈리는 이유를 문장의 꼴로 설명합니다.
- `check_table`에서 학습 후 예측이 바뀐 문장을 적고, 끝만 다른 세 문장(`닫았니`, `닫아`, `닫았어`)을 구분했는지 적습니다.